# DEST — Kaggle Limpio 20 + Seed Pool (20h GPU, sin desconexiones)

**Para Kaggle — no Colab**

* **Fix limpio:** `CollatzFix1/3` con `lexsort` 45k únicos (no 98% dup), clonado de `fed2d3c`
* **20 limpias:** `CIFAR-10` `stochastic` vs `collatz_v3` **10 seeds 300–309** (nuevo rango virgen, nunca usado, evita 42–61 y 200–209 bug) — 20 runs, ~70 min T4/P100, checkpoint a `/kaggle/working`
* **Seed pool:** además muestrea **10 seeds aleatorias del pool 0–1000** (sin repetir 300–309) para medir varianza inter-pool — total 20+20=40 runs si tienes 20h, si no, solo las 20 limpias
* **Salida:** `/kaggle/working/dest_kaggle_clean/*.json` + `kaggle_datasets` versionable


In [ ]:
# 0. Setup Kaggle — clona DEST fix, instala, verifica GPU
import os, sys, subprocess
print("🔧 Setup Kaggle...")
if os.path.exists("DEST"):
    subprocess.call(["rm","-rf","DEST"])
subprocess.check_call(["git","clone","https://github.com/starlyn2010/DEST.git"])
subprocess.check_call([sys.executable,"-m","pip","install","-e","DEST","--no-deps","-q"])
if "DEST/src" not in sys.path: sys.path.insert(0, "DEST/src")
import dest
sys.modules["dest_lib"]=dest
for sub in ["config","samplers","models","datasets","runner"]:
    try: m=__import__(f"dest.{sub}", fromlist=[sub]); sys.modules[f"dest_lib.{sub}"]=m
    except Exception as e: print(f"warn {sub}: {e}")
print("✅ DEST fix instalado")
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# Verificar fix
from dest.samplers import CollatzFix3Sampler
class Dummy: 
    def __len__(self): return 45000
s=CollatzFix3Sampler(Dummy(), seed=300); s.set_epoch(0); vals=s._collatz_step
print("✅ CollatzFix3Sampler con lexsort (45k únicos)")


In [ ]:
# 1. Config — 20 limpias (300–309) + 10 seed-pool aleatorias
from dest_lib.config import get_config
import random, numpy as np
random.seed(12345); np.random.seed(12345)
# Pool 0–1000 excluyendo 300–309 y 42–61 y 200–209 ya usados
used=set(list(range(42,62))+list(range(200,210))+list(range(300,310)))
pool=[s for s in range(0,1000) if s not in used]
seed_pool_random=sorted(random.sample(pool, 10))
print(f"Seed pool aleatorio: {seed_pool_random}")
config=get_config("PAPER")
config["datasets"]=["CIFAR10"]
config["samplers"]=["stochastic","collatz_v3"]
config["epochs"]=15
config["batch_size"]=128
config["lr"]=0.01
config["lr_schedule"]="cosine"
config["val_fraction"]=0.1
config["verbose"]=True
# Guardaremos en /kaggle/working para persistir entre sesiones Kaggle
config_clean=dict(config)
config_clean["seeds"]=list(range(300,310))
config_clean["output_dir"]="/kaggle/working/dest_kaggle_clean"
config_pool=dict(config)
config_pool["seeds"]=seed_pool_random
config_pool["output_dir"]="/kaggle/working/dest_kaggle_pool"

import os, json
for d in [config_clean["output_dir"], config_pool["output_dir"]]:
    os.makedirs(d, exist_ok=True)
    print(d, "->", len(os.listdir(d)), "existentes")
print(f"\nTotal limpio: {len(config_clean['seeds'])*2} runs")
print(f"Total pool: {len(config_pool['seeds'])*2} runs")
print(f"Total con ambos: 40 runs (~140 min P100, cabe en 20h)")


In [ ]:
# Pool desactivado para 20 limpias solo (ahorra 70 min)
# run_block(config_pool, "POOL aleatorio")


In [ ]:
# 3. Resumen y zip — 20 limpias
import glob, json, numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
for label,cfg in [("LIMPIO 300–309",config_clean), ("POOL",config_pool)]:
    pattern=os.path.join(cfg["output_dir"],"*.json")
    files=[f for f in glob.glob(pattern) if "sampler_name" in json.load(open(f))]
    print(f"\n{label}: {len(files)} JSONs")
    if not files: continue
    groups=defaultdict(list)
    for f in files:
        j=json.load(open(f)); groups[j["sampler_name"]].append(j)
    for s in ["stochastic","collatz_v3"]:
        arr=[j["final_test_acc"] for j in groups[s]]
        if arr: print(f"  {s:12s}: {np.mean(arr):.2f} ±{np.std(arr,ddof=1):.2f} n={len(arr)}")
    if "stochastic" in groups and "collatz_v3" in groups:
        stoch={j["seed"]:j["final_test_acc"] for j in groups["stochastic"]}
        v3={j["seed"]:j["final_test_acc"] for j in groups["collatz_v3"]}
        common=sorted(set(stoch)&set(v3))
        diffs=[v3[s]-stoch[s] for s in common]
        if diffs:
            from scipy import stats
            t,p=stats.ttest_rel([v3[s] for s in common],[stoch[s] for s in common])
            print(f"  V3 vs stoch: diff {np.mean(diffs):+.3f} p={p:.4f} d={np.mean(diffs)/np.std(diffs,ddof=1):.2f} gana {sum(d>0 for d in diffs)}/{len(diffs)}")
    # curva
    plt.figure(figsize=(7,3))
    for s in ["stochastic","collatz_v3"]:
        if s not in groups: continue
        arr=np.array([j["test_accs"] for j in groups[s]])
        plt.plot(range(1,16), arr.mean(0), label=s)
        plt.fill_between(range(1,16), arr.mean(0)-arr.std(0,ddof=1), arr.mean(0)+arr.std(0,ddof=1), alpha=0.15)
    plt.title(label); plt.xlabel("Época"); plt.ylabel("Test acc %"); plt.legend(); plt.grid(alpha=0.3)
    plt.savefig(os.path.join(cfg["output_dir"],"curvas.png"), dpi=200, bbox_inches="tight")
    plt.show()

# Zip final para descargar / versionar en Kaggle
import shutil
for cfg,label in [(config_clean,"LIMPIO_300_309"),(config_pool,"POOL")]:
    if os.path.exists(cfg["output_dir"]) and len(os.listdir(cfg["output_dir"]))>0:
        zipname=f"/kaggle/working/resultados_Kaggle_{label}.zip"
        shutil.make_archive(zipname.replace(".zip",""),'zip',cfg["output_dir"])
        print(f"✅ {zipname} {os.path.getsize(zipname)/1e6:.2f} MB")
